# 01. Данные и единая предобработка
Синтетика содержит повторяющиеся режимы, GARCH-инновации, чистый эталон и загрязнения. Все этапы используют один DataConfig; нормализатор обучается только на train.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "tsprep").is_dir() and (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the tsprep project")
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import matplotlib.pyplot as plt
import torch
torch.set_num_threads(2)


In [ ]:
from tsprep.config import DataConfig
from tsprep.data_preprocessing.synthetic import generate_series
from tsprep.data_preprocessing.datasets import prepare_and_window, forecast_pairs
from tsprep.data_preprocessing.preprocessing import inverse_series
series = generate_series(length=3000, seed=2026)
cfg = DataConfig(window_len=48, stride=2, log_transform=True)
prepared = prepare_and_window(series.values, cfg, regime=series.regime,
                              is_anomaly=series.is_anomaly, feature_names=["synthetic_price"])
print({s: getattr(prepared, s + "_windows").shape for s in ("train", "val", "test")})
print("train normalizer:", prepared.normalizer.state())
print("target: transformed observation at exact t +", cfg.forecast_horizon)


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8))
axes[0].plot(series.clean_values[:, 0], label="clean")
axes[0].plot(series.values[:, 0], label="observed", alpha=.6)
axes[0].legend()
axes[1].plot(prepared.raw_normalized[:, 0])
axes[1].set_title("Train-normalized log differences")
axes[2].plot(series.regime)
axes[2].set_title("Recurring regimes")
fig.tight_layout()
restored = inverse_series(prepared.raw_normalized, prepared.normalizer, prepared.y0,
                          cfg.seasonal_period, cfg.log_transform)
np.testing.assert_allclose(restored, series.values)


## Проверка реальных CSV
Подозрительные OHLC-строки не исправляются автоматически. Признаки выбираются явно; нулевой Volume не подходит для логарифмирования.


In [ ]:
from tsprep.data_preprocessing.quality import csv_quality_report
reports = {p.name: csv_quality_report(p) for p in sorted((ROOT / "data" / "raw").glob("*.csv"))}
reports
